In [4]:
import os
import cv2
import sys
from zipfile import ZipFile
from urllib.request import urlretrieve

############
import serial
import time

serialInst = serial.Serial()
serialInst.baudrate = 9600
serialInst.port = "COM3"


###OPENING PORT
if serialInst.is_open:
    print("Port was already open, closing it now...")
    serialInst.close()

try:
    serialInst.open()
    print("Port opened successfully!")
    time.sleep(2) # Vital: give Arduino time to reboot

except Exception as e:
    print(f"Error: {e}")

last_send_time = time.time()
send_interval = 0.1  # Send data every 0.1 seconds (10 times per sec)
#############


# ======================== Downloading Assets =========================
URL = r"https://www.dropbox.com/s/efitgt363ada95a/opencv_bootcamp_assets_12.zip?dl=1"

asset_zip_path = os.path.join(os.getcwd(), "opencv_bootcamp_assets_12.zip")

prototxt_path = "deploy.prototxt"
model_path = "res10_300x300_ssd_iter_140000_fp16.caffemodel"

def download_and_unzip(url, save_path):
    print("Downloading assets...", end="")
    urlretrieve(url, save_path)

    with ZipFile(save_path) as z:
        z.extractall(os.path.dirname(save_path))

    os.remove(save_path)  # delete zip after extracting
    print("Done")

# Only download if model files are missing
if not (os.path.exists(prototxt_path) and os.path.exists(model_path)):
    download_and_unzip(URL, asset_zip_path)
# ====================================================================


#s = 0
s = "http://172.20.10.11:81/stream"

source = cv2.VideoCapture(s)

win_name = "Camera Preview"
cv2.namedWindow(win_name, cv2.WINDOW_NORMAL)

net = cv2.dnn.readNetFromCaffe("deploy.prototxt", "res10_300x300_ssd_iter_140000_fp16.caffemodel")
# Model parameters
in_width = 300
in_height = 300
mean = [104, 117, 123]
conf_threshold = 0.7

while cv2.waitKey(1) != 27:
    
    has_frame, frame = source.read()
    if not has_frame:
        break
    frame = cv2.flip(frame, 1)
    frame_height = frame.shape[0]
    frame_width = frame.shape[1]

    # Create a 4D blob from a frame.
    blob = cv2.dnn.blobFromImage(frame, 1.0, (in_width, in_height), mean, swapRB=False, crop=False)
    # Run a model
    net.setInput(blob)
    detections = net.forward()

    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > conf_threshold:
            x_top_left = int(detections[0, 0, i, 3] * frame_width)
            y_top_left = int(detections[0, 0, i, 4] * frame_height)
            x_bottom_right  = int(detections[0, 0, i, 5] * frame_width)
            y_bottom_right  = int(detections[0, 0, i, 6] * frame_height)

            cv2.rectangle(frame, (x_top_left, y_top_left), (x_bottom_right, y_bottom_right), (0, 255, 0))
            label = "Confidence: %.4f" % confidence
            label_size, base_line = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)

            #cv2.rectangle(
                #frame,
                #(x_top_left, y_top_left - label_size[1]),
                #(x_top_left + label_size[0], y_top_left + base_line),
                #(255, 255, 255),
                #cv2.FILLED,
            #)
            #cv2.putText(frame, label, (x_top_left, y_top_left), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0))

            #########################
            x_middle = int((x_top_left + x_bottom_right) / 2)
            y_middle = int((y_top_left + y_bottom_right) / 2)

            #frame_width
            #frame_height

            # NON-BLOCKING SERIAL SEND
            current_time = time.time()
            if (current_time - last_send_time) > send_interval:
                msg = f"{x_middle},{y_middle},{frame_width},{frame_height}\n"
                serialInst.write(msg.encode('utf-8'))
                last_send_time = current_time # Reset the timer

            # NON-BLOCKING SERIAL READ (Check every frame)
            if serialInst.in_waiting > 0:
                try:
                    response = serialInst.readline().decode('utf-8').strip()
                    print(f"ARDUINO: {response}")
                except:
                    pass # Ignore decoding glitches

            ##########################
        

    cv2.imshow(win_name, frame)

        

source.release()
cv2.destroyWindow(win_name)
serialInst.close()

ModuleNotFoundError: No module named 'cv2'

In [ ]:
#TEST FOR PYSERIAL CONNECTION

import serial
import time

serialInst = serial.Serial()
serialInst.baudrate = 9600
serialInst.port = "COM3"

# CHECK IF OPEN FIRST
if serialInst.is_open:
    print("Port was already open, closing it now...")
    serialInst.close()

try:
    serialInst.open()
    print("Port opened successfully!")
    time.sleep(2) # Vital: give Arduino time to reboot
    
    while True:
        
        command = input("input: ")
        if command == 'exit':
            break
            
        serialInst.write(command.encode('utf-8'))
        time.sleep(1.5) 
    
        # 3. Read the response
        if serialInst.in_waiting > 0: # Check if there is data waiting to be read
            raw_data = serialInst.readline()
            # Decode bytes to string and clean up whitespace
            response = raw_data.decode('utf-8').strip() 
            print(f"Response: {response}")





except Exception as e:
    print(f"Error: {e}")

finally:
    serialInst.close()
    print("Port closed safely.")

Port opened successfully!


input:  exit


Port closed safely.
